In [1]:
import numpy as np
import spacy
import warnings
import os
import time
import pandas as pd
from dotenv import load_dotenv
from scipy.optimize import minimize
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import logging

# --- Together AI Client ---
from together import Together

# Suppress warnings for clean terminal output
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

# --- Qiskit Imports ---
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_aer.primitives import Sampler as LocalSampler

# --- Configuration ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Define the old CSV to read from and the new CSV to write to
OLD_CSV_FILENAME = "qrag_telemetry_N150_run_1783611471_final.csv"  # <-- UPDATE THIS TO YOUR PREVIOUS RUN'S CSV
RUN_TIMESTAMP = int(time.time())
CSV_FILENAME = f"qrag_telemetry_Updated_run_{RUN_TIMESTAMP}.csv"

# Load environment variables
load_dotenv()

# ==============================================================================
# DATASET PLACEHOLDER (NEW AGENT-PATIENT INVERSION SENTENCES)
# ==============================================================================
NEW_DATABASE = [
  {
    "class": "Agent-Patient Inversion",
    "text": "The sliced bread cut the serrated bread knife.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sliced bread",
    "conflict": "the serrated bread knife"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The whisked eggs beat the stainless wire whisk.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the whisked eggs",
    "conflict": "the stainless wire whisk"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The strained pasta drained the metal colander.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the strained pasta",
    "conflict": "the metal colander"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sifted flour filtered the mesh flour sifter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sifted flour",
    "conflict": "the mesh flour sifter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened can punctured the manual can opener.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened can",
    "conflict": "the manual can opener"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The uncorked wine pulled the winged corkscrew.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the uncorked wine",
    "conflict": "the winged corkscrew"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The baked cookies heated the flat baking sheet.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the baked cookies",
    "conflict": "the flat baking sheet"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The toasted bread browned the popup slot toaster.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the toasted bread",
    "conflict": "the popup slot toaster"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mixed batter stirred the electric stand mixer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mixed batter",
    "conflict": "the electric stand mixer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ground coffee crushed the burr coffee grinder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ground coffee",
    "conflict": "the burr coffee grinder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The juiced oranges squeezed the plastic citrus juicer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the juiced oranges",
    "conflict": "the plastic citrus juicer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chopped nuts diced the electric food processor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chopped nuts",
    "conflict": "the electric food processor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The flipped pancakes tossed the silicone spatula.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the flipped pancakes",
    "conflict": "the silicone spatula"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scooped ice cream dug the metal ice cream scoop.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scooped ice cream",
    "conflict": "the metal ice cream scoop"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled dough flattened the heavy wooden rolling pin.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled dough",
    "conflict": "the heavy wooden rolling pin"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The grated cheese shredded the rotary cheese grater.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the grated cheese",
    "conflict": "the rotary cheese grater"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The peeled potato skinned the stainless vegetable peeler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the peeled potato",
    "conflict": "the stainless vegetable peeler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The carved turkey sliced the electric carving knife.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the carved turkey",
    "conflict": "the electric carving knife"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The brewed tea steeped the mesh tea infuser.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the brewed tea",
    "conflict": "the mesh tea infuser"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shaved ice scraped the heavy ice shaver.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shaved ice",
    "conflict": "the heavy ice shaver"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted butter heated the copper melting pot.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted butter",
    "conflict": "the copper melting pot"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The tenderized meat pounded the spiked meat mallet.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the tenderized meat",
    "conflict": "the spiked meat mallet"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spiralized zucchini twisted the plastic vegetable spiralizer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spiralized zucchini",
    "conflict": "the plastic vegetable spiralizer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The microplaned zest zested the steel microplane grater.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the microplaned zest",
    "conflict": "the steel microplane grater"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cored apple pierced the tubular apple corer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cored apple",
    "conflict": "the tubular apple corer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pitted cherries punched the metal cherry pitter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pitted cherries",
    "conflict": "the metal cherry pitter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The crushed garlic squeezed the heavy garlic press.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the crushed garlic",
    "conflict": "the heavy garlic press"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The hulled strawberries plucked the sharp strawberry huller.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the hulled strawberries",
    "conflict": "the sharp strawberry huller"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The portioned batter dropped the spring-loaded cookie scoop.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the portioned batter",
    "conflict": "the spring-loaded cookie scoop"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The whipped cream aerated the pressurized cream dispenser.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the whipped cream",
    "conflict": "the pressurized cream dispenser"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The boiled eggs timed the analog egg timer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the boiled eggs",
    "conflict": "the analog egg timer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The poached eggs shaped the silicone egg poacher.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the poached eggs",
    "conflict": "the silicone egg poacher"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed vegetables boiled the bamboo vegetable steamer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed vegetables",
    "conflict": "the bamboo vegetable steamer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The wok-fried noodles tossed the carbon steel wok.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the wok-fried noodles",
    "conflict": "the carbon steel wok"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The deep-fried potatoes submerged the wire deep fryer basket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the deep-fried potatoes",
    "conflict": "the wire deep fryer basket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The seared scallops blackened the cast iron skillet.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the seared scallops",
    "conflict": "the cast iron skillet"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The slow-cooked stew simmered the ceramic slow cooker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the slow-cooked stew",
    "conflict": "the ceramic slow cooker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pressure-cooked roast contained the stovetop pressure cooker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pressure-cooked roast",
    "conflict": "the stovetop pressure cooker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The air-fried wings crisped the digital air fryer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the air-fried wings",
    "conflict": "the digital air fryer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The smoked brisket flavored the wood pellet smoker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the smoked brisket",
    "conflict": "the wood pellet smoker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The grilled burgers seared the outdoor gas grill.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the grilled burgers",
    "conflict": "the outdoor gas grill"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chilled wine cooled the thermal wine cooler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chilled wine",
    "conflict": "the thermal wine cooler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The frozen popsicles molded the plastic popsicle molds.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the frozen popsicles",
    "conflict": "the plastic popsicle molds"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The decanted wine aerated the crystal wine decanter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the decanted wine",
    "conflict": "the crystal wine decanter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The served cake lifted the triangular cake server.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the served cake",
    "conflict": "the triangular cake server"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The poured soup ladled the deep soup ladle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the poured soup",
    "conflict": "the deep soup ladle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steeped coffee pressed the glass French press.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steeped coffee",
    "conflict": "the glass French press"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The brewed espresso extracted the heavy espresso machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the brewed espresso",
    "conflict": "the heavy espresso machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sliced pizza rolled the circular pizza cutter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sliced pizza",
    "conflict": "the circular pizza cutter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked nuts snapped the heavy metal nutcracker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked nuts",
    "conflict": "the heavy metal nutcracker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The swept floor brushed the straw bristle broom.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the swept floor",
    "conflict": "the straw bristle broom"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mopped tile washed the sponge floor mop.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mopped tile",
    "conflict": "the sponge floor mop"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dusted shelf wiped the ostrich feather duster.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dusted shelf",
    "conflict": "the ostrich feather duster"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The wiped glass cleaned the blue microfiber cloth.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the wiped glass",
    "conflict": "the blue microfiber cloth"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scrubbed tub scoured the stiff bristle scrub brush.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scrubbed tub",
    "conflict": "the stiff bristle scrub brush"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The vacuumed carpet sucked the upright vacuum cleaner.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the vacuumed carpet",
    "conflict": "the upright vacuum cleaner"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The squeegeed window scraped the rubber window squeegee.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the squeegeed window",
    "conflict": "the rubber window squeegee"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The raked leaves gathered the wide plastic leaf rake.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the raked leaves",
    "conflict": "the wide plastic leaf rake"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The polished wood rubbed the lemon furniture polish.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the polished wood",
    "conflict": "the lemon furniture polish"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The disinfected counter sprayed the antibacterial cleaning spray.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the disinfected counter",
    "conflict": "the antibacterial cleaning spray"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The washed window wiped the wet window sponge.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the washed window",
    "conflict": "the wet window sponge"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ironed shirt pressed the hot steam iron.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ironed shirt",
    "conflict": "the hot steam iron"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed garment smoothed the handheld garment steamer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed garment",
    "conflict": "the handheld garment steamer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The folded clothes creased the plastic laundry folder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the folded clothes",
    "conflict": "the plastic laundry folder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The lint-rolled suit peeled the sticky lint roller.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the lint-rolled suit",
    "conflict": "the sticky lint roller"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sorted laundry divided the woven laundry hamper.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sorted laundry",
    "conflict": "the woven laundry hamper"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The washed dishes cleansed the scented dish soap.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the washed dishes",
    "conflict": "the scented dish soap"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dried plates wiped the cotton dish towel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dried plates",
    "conflict": "the cotton dish towel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scrubbed pot scratched the steel wool scouring pad.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scrubbed pot",
    "conflict": "the steel wool scouring pad"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The unblocked drain plunged the rubber toilet plunger.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the unblocked drain",
    "conflict": "the rubber toilet plunger"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snaked pipe cleared the flexible plumbing snake.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snaked pipe",
    "conflict": "the flexible plumbing snake"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sanitized toilet bleached the nylon toilet brush.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sanitized toilet",
    "conflict": "the nylon toilet brush"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The swept driveway blew the gas-powered leaf blower.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the swept driveway",
    "conflict": "the gas-powered leaf blower"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pressure-washed deck sprayed the motorized pressure washer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pressure-washed deck",
    "conflict": "the motorized pressure washer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shoveled snow scooped the wide snow shovel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shoveled snow",
    "conflict": "the wide snow shovel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted ice salted the chemical rock salt.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted ice",
    "conflict": "the chemical rock salt"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The heated room warmed the electric space heater.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the heated room",
    "conflict": "the electric space heater"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cooled air chilled the window air conditioner.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cooled air",
    "conflict": "the window air conditioner"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The humidified bedroom misted the ultrasonic cool mist humidifier.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the humidified bedroom",
    "conflict": "the ultrasonic cool mist humidifier"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dehumidified basement dried the heavy compressor dehumidifier.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dehumidified basement",
    "conflict": "the heavy compressor dehumidifier"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The purified air filtered the HEPA room purifier.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the purified air",
    "conflict": "the HEPA room purifier"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The circulated air blew the oscillating pedestal fan.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the circulated air",
    "conflict": "the oscillating pedestal fan"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The exhausted smoke vented the overhead range hood.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the exhausted smoke",
    "conflict": "the overhead range hood"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The illuminated hall lit the LED light bulb.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the illuminated hall",
    "conflict": "the LED light bulb"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The darkened room blocked the heavy blackout curtains.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the darkened room",
    "conflict": "the heavy blackout curtains"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The secured door locked the solid brass deadbolt.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the secured door",
    "conflict": "the solid brass deadbolt"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The silenced alarm pressed the digital snooze button.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the silenced alarm",
    "conflict": "the digital snooze button"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The recorded show programmed the digital video recorder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the recorded show",
    "conflict": "the digital video recorder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The played music spun the vintage record player.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the played music",
    "conflict": "the vintage record player"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The tuned radio twisted the analog tuning dial.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the tuned radio",
    "conflict": "the analog tuning dial"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The displayed image glowed the flat OLED television.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the displayed image",
    "conflict": "the flat OLED television"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The projected screen beamed the home theater projector.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the projected screen",
    "conflict": "the home theater projector"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charged phone plugged the braided USB cable.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charged phone",
    "conflict": "the braided USB cable"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The powered laptop connected the heavy charging brick.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the powered laptop",
    "conflict": "the heavy charging brick"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The protected outlet tripped the electrical surge protector.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the protected outlet",
    "conflict": "the electrical surge protector"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The measured weight displayed the digital bathroom scale.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the measured weight",
    "conflict": "the digital bathroom scale"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The timed workout counted the digital stopwatch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the timed workout",
    "conflict": "the digital stopwatch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The tracked steps sensed the wearable fitness tracker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the tracked steps",
    "conflict": "the wearable fitness tracker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The monitored heartrate strapped the elastic chest monitor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the monitored heartrate",
    "conflict": "the elastic chest monitor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The massaged back vibrated the handheld percussive massager.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the massaged back",
    "conflict": "the handheld percussive massager"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The nailed wood struck the heavy claw hammer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the nailed wood",
    "conflict": "the heavy claw hammer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The screwed hinge twisted the manual Phillips screwdriver.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the screwed hinge",
    "conflict": "the manual Phillips screwdriver"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sawed log cut the motorized chain saw.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sawed log",
    "conflict": "the motorized chain saw"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sanded board smoothed the electric orbital sander.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sanded board",
    "conflict": "the electric orbital sander"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The painted wall rolled the fuzzy paint roller.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the painted wall",
    "conflict": "the fuzzy paint roller"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The glued joint bonded the liquid wood glue.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the glued joint",
    "conflict": "the liquid wood glue"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The clamped table squeezed the cast iron C-clamp.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the clamped table",
    "conflict": "the cast iron C-clamp"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The soldered wire melted the hot soldering iron.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the soldered wire",
    "conflict": "the hot soldering iron"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The measured room extended the retractable tape measure.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the measured room",
    "conflict": "the retractable tape measure"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The routed edge carved the high-speed wood router.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the routed edge",
    "conflict": "the high-speed wood router"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The planed door shaved the sharp block plane.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the planed door",
    "conflict": "the sharp block plane"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bolted engine torqued the ratcheting socket wrench.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bolted engine",
    "conflict": "the ratcheting socket wrench"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The welded pipe fused the bright welding torch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the welded pipe",
    "conflict": "the bright welding torch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chiseled stone broke the steel cold chisel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chiseled stone",
    "conflict": "the steel cold chisel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The filed metal scraped the coarse bastard file.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the filed metal",
    "conflict": "the coarse bastard file"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drilled wood penetrated the cordless power drill.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drilled wood",
    "conflict": "the cordless power drill"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The leveled shelf balanced the yellow bubble level.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the leveled shelf",
    "conflict": "the yellow bubble level"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The marked board lined the blue chalk line.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the marked board",
    "conflict": "the blue chalk line"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The gripped pipe grabbed the adjustable pipe wrench.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the gripped pipe",
    "conflict": "the adjustable pipe wrench"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut wire snipped the diagonal wire cutters.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut wire",
    "conflict": "the diagonal wire cutters"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stripped wire exposed the manual wire strippers.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stripped wire",
    "conflict": "the manual wire strippers"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The crimped connector squeezed the metal crimping tool.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the crimped connector",
    "conflict": "the metal crimping tool"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stapled insulation punched the heavy staple gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stapled insulation",
    "conflict": "the heavy staple gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The riveted sheet popped the manual pop riveter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the riveted sheet",
    "conflict": "the manual pop riveter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The painted trim brushed the angled sash brush.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the painted trim",
    "conflict": "the angled sash brush"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scraped paint peeled the rigid putty knife.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scraped paint",
    "conflict": "the rigid putty knife"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The caulked seam squeezed the dripless caulk gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the caulked seam",
    "conflict": "the dripless caulk gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The tightened nut turned the crescent adjustable wrench.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the tightened nut",
    "conflict": "the crescent adjustable wrench"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The loosened bolt grabbed the locking vise-grip pliers.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the loosened bolt",
    "conflict": "the locking vise-grip pliers"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pryed nail lifted the steel pry bar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pryed nail",
    "conflict": "the steel pry bar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut glass scored the diamond-tipped glass cutter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut glass",
    "conflict": "the diamond-tipped glass cutter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted solder wicked the copper desoldering braid.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted solder",
    "conflict": "the copper desoldering braid"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The insulated wire wrapped the black electrical tape.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the insulated wire",
    "conflict": "the black electrical tape"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The lubricated hinge sprayed the aerosol penetrating oil.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the lubricated hinge",
    "conflict": "the aerosol penetrating oil"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The polished metal buffed the cotton buffing wheel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the polished metal",
    "conflict": "the cotton buffing wheel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sharpened blade ground the heavy bench grinder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sharpened blade",
    "conflict": "the heavy bench grinder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The engraved tag scratched the rotary engraving tool.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the engraved tag",
    "conflict": "the rotary engraving tool"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut tile snapped the manual tile cutter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut tile",
    "conflict": "the manual tile cutter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mixed concrete churned the portable cement mixer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mixed concrete",
    "conflict": "the portable cement mixer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The poured driveway floated the magnesium bull float.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the poured driveway",
    "conflict": "the magnesium bull float"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The smoothed plaster troweled the flat masonry trowel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the smoothed plaster",
    "conflict": "the flat masonry trowel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The framed wall shot the pneumatic framing nailer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the framed wall",
    "conflict": "the pneumatic framing nailer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The secured drywall screwed the electric drywall gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the secured drywall",
    "conflict": "the electric drywall gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The taped seam covered the paper drywall tape.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the taped seam",
    "conflict": "the paper drywall tape"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mudded joint spread the wide taping knife.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mudded joint",
    "conflict": "the wide taping knife"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The hoisted engine lifted the hydraulic engine hoist.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the hoisted engine",
    "conflict": "the hydraulic engine hoist"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The supported car jacked the steel jack stands.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the supported car",
    "conflict": "the steel jack stands"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The inflated tire pumped the stationary air compressor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the inflated tire",
    "conflict": "the stationary air compressor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The greased fitting injected the heavy grease gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the greased fitting",
    "conflict": "the heavy grease gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The jump-started battery clamped the thick jumper cables.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the jump-started battery",
    "conflict": "the thick jumper cables"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stapled document punched the desktop metal stapler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stapled document",
    "conflict": "the desktop metal stapler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The hole-punched paper pressed the mechanical three-hole punch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the hole-punched paper",
    "conflict": "the mechanical three-hole punch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The erased mistake rubbed the pink rubber eraser.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the erased mistake",
    "conflict": "the pink rubber eraser"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The highlighted text colored the neon yellow highlighter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the highlighted text",
    "conflict": "the neon yellow highlighter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut cardboard scissored the sharp craft scissors.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut cardboard",
    "conflict": "the sharp craft scissors"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The glued poster adhered the sticky glue stick.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the glued poster",
    "conflict": "the sticky glue stick"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stamped envelope inked the custom rubber stamp.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stamped envelope",
    "conflict": "the custom rubber stamp"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shredded document sliced the noisy paper shredder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shredded document",
    "conflict": "the noisy paper shredder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The taped box sealed the handheld packaging tape dispenser.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the taped box",
    "conflict": "the handheld packaging tape dispenser"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The clipped files pinched the black binder clip.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the clipped files",
    "conflict": "the black binder clip"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pinned notice stabbed the sharp plastic pushpin.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pinned notice",
    "conflict": "the sharp plastic pushpin"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bound report clamped the plastic comb binding machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bound report",
    "conflict": "the plastic comb binding machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The laminated sign melted the thermal pouch laminator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the laminated sign",
    "conflict": "the thermal pouch laminator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The printed page inked the desktop inkjet printer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the printed page",
    "conflict": "the desktop inkjet printer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scanned receipt digitized the automatic document feeder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scanned receipt",
    "conflict": "the automatic document feeder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The faxed contract transmitted the analog fax machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the faxed contract",
    "conflict": "the analog fax machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The projected presentation beamed the digital classroom projector.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the projected presentation",
    "conflict": "the digital classroom projector"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The calculated sum tallied the solar desktop calculator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the calculated sum",
    "conflict": "the solar desktop calculator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The weighed letter balanced the digital postage scale.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the weighed letter",
    "conflict": "the digital postage scale"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The metered mail stamped the electronic postage meter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the metered mail",
    "conflict": "the electronic postage meter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The typed letter clicked the tactile mechanical keyboard.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the typed letter",
    "conflict": "the tactile mechanical keyboard"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The clicked link tracked the wireless optical mouse.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the clicked link",
    "conflict": "the wireless optical mouse"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The tapped screen registered the active stylus pen.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the tapped screen",
    "conflict": "the active stylus pen"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The written note scribbled the blue ballpoint pen.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the written note",
    "conflict": "the blue ballpoint pen"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drafted blueprint marked the mechanical drafting pencil.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drafted blueprint",
    "conflict": "the mechanical drafting pencil"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drawn circle rotated the steel drafting compass.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drawn circle",
    "conflict": "the steel drafting compass"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The measured line guided the clear plastic ruler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the measured line",
    "conflict": "the clear plastic ruler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The read book flipped the printed paper pages.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the read book",
    "conflict": "the printed paper pages"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bookmarked page clipped the magnetic page marker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bookmarked page",
    "conflict": "the magnetic page marker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The organized files sorted the expanding file folder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the organized files",
    "conflict": "the expanding file folder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The labeled box printed the thermal label maker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the labeled box",
    "conflict": "the thermal label maker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened mail slit the metallic letter opener.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened mail",
    "conflict": "the metallic letter opener"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The filed document stored the steel filing cabinet.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the filed document",
    "conflict": "the steel filing cabinet"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stored data saved the portable USB flash drive.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stored data",
    "conflict": "the portable USB flash drive"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The backed-up system wrote the external hard drive.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the backed-up system",
    "conflict": "the external hard drive"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The networked office connected the gigabit ethernet switch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the networked office",
    "conflict": "the gigabit ethernet switch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broadcasted signal transmitted the wireless internet router.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broadcasted signal",
    "conflict": "the wireless internet router"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The displayed chart lit the external LCD monitor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the displayed chart",
    "conflict": "the external LCD monitor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The recorded meeting captured the omnidirectional conference microphone.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the recorded meeting",
    "conflict": "the omnidirectional conference microphone"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The played presentation echoed the desktop computer speakers.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the played presentation",
    "conflict": "the desktop computer speakers"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scheduled appointment alerted the digital calendar app.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scheduled appointment",
    "conflict": "the digital calendar app"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sent email routed the enterprise mail server.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sent email",
    "conflict": "the enterprise mail server"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The encrypted message scrambled the secure cryptographic key.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the encrypted message",
    "conflict": "the secure cryptographic key"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The unlocked device scanned the optical fingerprint reader.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the unlocked device",
    "conflict": "the optical fingerprint reader"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The authenticated login checked the physical security token.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the authenticated login",
    "conflict": "the physical security token"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charged tablet drained the inductive charging pad.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charged tablet",
    "conflict": "the inductive charging pad"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cleaned monitor wiped the disposable screen wipe.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cleaned monitor",
    "conflict": "the disposable screen wipe"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cooled laptop spun the external cooling pad.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cooled laptop",
    "conflict": "the external cooling pad"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The supported wrists rested the gel keyboard wrist rest.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the supported wrists",
    "conflict": "the gel keyboard wrist rest"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The adjusted chair reclined the ergonomic office chair.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the adjusted chair",
    "conflict": "the ergonomic office chair"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The combed hair separated the plastic pocket comb.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the combed hair",
    "conflict": "the plastic pocket comb"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The clipped nails snapped the stainless nail clippers.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the clipped nails",
    "conflict": "the stainless nail clippers"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The filed fingernails sanded the rough emery board.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the filed fingernails",
    "conflict": "the rough emery board"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shaved legs scraped the disposable safety razor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shaved legs",
    "conflict": "the disposable safety razor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The curled hair wrapped the heated curling iron.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the curled hair",
    "conflict": "the heated curling iron"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The straightened hair clamped the ceramic hair straightener.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the straightened hair",
    "conflict": "the ceramic hair straightener"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The blow-dried hair blasted the ionic hair dryer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the blow-dried hair",
    "conflict": "the ionic hair dryer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The tweezed eyebrows plucked the angled steel tweezers.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the tweezed eyebrows",
    "conflict": "the angled steel tweezers"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The brushed teeth scrubbed the manual soft toothbrush.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the brushed teeth",
    "conflict": "the manual soft toothbrush"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The flossed gums cleaned the waxed dental floss.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the flossed gums",
    "conflict": "the waxed dental floss"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The injected vaccine pierced the thin hypodermic needle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the injected vaccine",
    "conflict": "the thin hypodermic needle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drawn blood vacuumed the sterile plastic syringe.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drawn blood",
    "conflict": "the sterile plastic syringe"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stitched wound threaded the curved surgical needle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stitched wound",
    "conflict": "the curved surgical needle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bandaged cut wrapped the sterile gauze roll.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bandaged cut",
    "conflict": "the sterile gauze roll"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The listened heart thumped the binaural acoustic stethoscope.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the listened heart",
    "conflict": "the binaural acoustic stethoscope"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The measured fever beeped the digital oral thermometer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the measured fever",
    "conflict": "the digital oral thermometer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The weighed patient balanced the mechanical medical scale.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the weighed patient",
    "conflict": "the mechanical medical scale"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun plasma centrifuged the electric laboratory centrifuge.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun plasma",
    "conflict": "the electric laboratory centrifuge"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pipetted serum transferred the adjustable micropipette.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pipetted serum",
    "conflict": "the adjustable micropipette"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The magnified cells focused the optical compound microscope.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the magnified cells",
    "conflict": "the optical compound microscope"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The diagnosed bone x-rayed the digital radiography scanner.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the diagnosed bone",
    "conflict": "the digital radiography scanner"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scanned brain imaged the loud MRI machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scanned brain",
    "conflict": "the loud MRI machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The monitored pulse beeped the optical pulse oximeter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the monitored pulse",
    "conflict": "the optical pulse oximeter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The checked blood pressure pumped the inflatable sphygmomanometer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the checked blood pressure",
    "conflict": "the inflatable sphygmomanometer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stabilized neck supported the rigid cervical collar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stabilized neck",
    "conflict": "the rigid cervical collar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The braced knee hinged the flexible neoprene knee brace.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the braced knee",
    "conflict": "the flexible neoprene knee brace"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The casted arm hardened the activated fiberglass cast.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the casted arm",
    "conflict": "the activated fiberglass cast"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The supported walk leaned the aluminum medical crutches.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the supported walk",
    "conflict": "the aluminum medical crutches"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pushed patient rolled the folding manual wheelchair.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pushed patient",
    "conflict": "the folding manual wheelchair"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cleared airway suctioned the medical vacuum aspirator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cleared airway",
    "conflict": "the medical vacuum aspirator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The oxygenated patient breathed the transparent nasal cannula.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the oxygenated patient",
    "conflict": "the transparent nasal cannula"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ventilated lungs pumped the mechanical hospital ventilator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ventilated lungs",
    "conflict": "the mechanical hospital ventilator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shocked heart defibrillated the automated external defibrillator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shocked heart",
    "conflict": "the automated external defibrillator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The clamped artery locked the surgical steel hemostat.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the clamped artery",
    "conflict": "the surgical steel hemostat"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cauterized tissue burned the handheld electrocautery pen.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cauterized tissue",
    "conflict": "the handheld electrocautery pen"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The intubated trachea guided the flexible endotracheal tube.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the intubated trachea",
    "conflict": "the flexible endotracheal tube"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drained fluid flowed the sterile surgical catheter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drained fluid",
    "conflict": "the sterile surgical catheter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut tissue snipped the curved surgical scissors.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut tissue",
    "conflict": "the curved surgical scissors"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The retracted skin pulled the metal surgical retractor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the retracted skin",
    "conflict": "the metal surgical retractor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The illuminated surgery shone the overhead surgical light.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the illuminated surgery",
    "conflict": "the overhead surgical light"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sterilized instruments steamed the high-pressure medical autoclave.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sterilized instruments",
    "conflict": "the high-pressure medical autoclave"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cultured bacteria grew the round plastic petri dish.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cultured bacteria",
    "conflict": "the round plastic petri dish"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The swiped sample collected the sterile cotton swab.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the swiped sample",
    "conflict": "the sterile cotton swab"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The heated reaction warmed the precise laboratory hotplate.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the heated reaction",
    "conflict": "the precise laboratory hotplate"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The titrated chemical dripped the graduated glass burette.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the titrated chemical",
    "conflict": "the graduated glass burette"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mixed solution swirled the magnetic stir bar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mixed solution",
    "conflict": "the magnetic stir bar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The measured volume filled the graduated glass cylinder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the measured volume",
    "conflict": "the graduated glass cylinder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The weighed powder depressed the analytical laboratory microbalance.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the weighed powder",
    "conflict": "the analytical laboratory microbalance"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The incubated cells warmed the regulated cell incubator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the incubated cells",
    "conflict": "the regulated cell incubator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sequenced DNA processed the automated genetic sequencer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sequenced DNA",
    "conflict": "the automated genetic sequencer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The watered plants showered the plastic watering can.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the watered plants",
    "conflict": "the plastic watering can"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pruned branch snipped the bypass pruning shears.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pruned branch",
    "conflict": "the bypass pruning shears"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dug hole lifted the pointed steel shovel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dug hole",
    "conflict": "the pointed steel shovel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The trimmed hedge sheared the electric hedge trimmer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the trimmed hedge",
    "conflict": "the electric hedge trimmer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chopped firewood split the heavy splitting maul.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chopped firewood",
    "conflict": "the heavy splitting maul"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The caught fish hooked the barbed treble hook.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the caught fish",
    "conflict": "the barbed treble hook"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The netted butterfly trapped the mesh butterfly net.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the netted butterfly",
    "conflict": "the mesh butterfly net"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The braked car clamped the ceramic brake pads.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the braked car",
    "conflict": "the ceramic brake pads"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steered ship rotated the wooden ship wheel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steered ship",
    "conflict": "the wooden ship wheel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rowed boat paddled the long wooden oars.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rowed boat",
    "conflict": "the long wooden oars"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sailed yacht caught the triangular canvas jib.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sailed yacht",
    "conflict": "the triangular canvas jib"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pedaled bike turned the aluminum bicycle pedals.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pedaled bike",
    "conflict": "the aluminum bicycle pedals"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The anchored boat dropped the heavy mushroom anchor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the anchored boat",
    "conflict": "the heavy mushroom anchor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The accelerated motorcycle twisted the rubber throttle grip.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the accelerated motorcycle",
    "conflict": "the rubber throttle grip"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shifted gears moved the manual transmission stick.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shifted gears",
    "conflict": "the manual transmission stick"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The wiped windshield cleared the rubber wiper blades.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the wiped windshield",
    "conflict": "the rubber wiper blades"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The honked horn sounded the steering wheel button.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the honked horn",
    "conflict": "the steering wheel button"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The fueled tank pumped the heavy gasoline nozzle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the fueled tank",
    "conflict": "the heavy gasoline nozzle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charged vehicle plugged the high-voltage charging cable.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charged vehicle",
    "conflict": "the high-voltage charging cable"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The locked car clicked the remote key fob.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the locked car",
    "conflict": "the remote key fob"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The navigated route displayed the dashboard GPS unit.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the navigated route",
    "conflict": "the dashboard GPS unit"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The hit baseball swung the composite aluminum bat.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the hit baseball",
    "conflict": "the composite aluminum bat"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The kicked soccer ball booted the spiked leather cleat.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the kicked soccer ball",
    "conflict": "the spiked leather cleat"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The caught flyball squeezed the oversized outfielder glove.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the caught flyball",
    "conflict": "the oversized outfielder glove"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shot puck slapped the carbon fiber hockey stick.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shot puck",
    "conflict": "the carbon fiber hockey stick"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spiked volleyball slapped the taut volleyball net.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spiked volleyball",
    "conflict": "the taut volleyball net"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The driven golf ball struck the titanium oversized driver.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the driven golf ball",
    "conflict": "the titanium oversized driver"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The lifted weights pressed the knurled Olympic barbell.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the lifted weights",
    "conflict": "the knurled Olympic barbell"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The punched bag struck the padded leather boxing gloves.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the punched bag",
    "conflict": "the padded leather boxing gloves"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The served tennis ball smashed the strung tennis racket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the served tennis ball",
    "conflict": "the strung tennis racket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bowled strike knocked the heavy urethane bowling ball.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bowled strike",
    "conflict": "the heavy urethane bowling ball"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shot arrow launched the fiberglass recurve bow.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shot arrow",
    "conflict": "the fiberglass recurve bow"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The caught pass gripped the sticky receiver gloves.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the caught pass",
    "conflict": "the sticky receiver gloves"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scaled wall grabbed the synthetic climbing hold.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scaled wall",
    "conflict": "the synthetic climbing hold"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The skied slope carved the waxed downhill skis.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the skied slope",
    "conflict": "the waxed downhill skis"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snowboarded rail slid the flexible freestyle snowboard.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snowboarded rail",
    "conflict": "the flexible freestyle snowboard"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The surfed wave rode the fiberglass longboard surfboard.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the surfed wave",
    "conflict": "the fiberglass longboard surfboard"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The strummed chord vibrated the acoustic guitar strings.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the strummed chord",
    "conflict": "the acoustic guitar strings"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bowed note rubbed the horsehair violin bow.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bowed note",
    "conflict": "the horsehair violin bow"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The struck cymbal crashed the wooden hickory drumstick.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the struck cymbal",
    "conflict": "the wooden hickory drumstick"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The blown melody tooted the brass trumpet mouthpiece.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the blown melody",
    "conflict": "the brass trumpet mouthpiece"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pressed keys played the grand piano ivory.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pressed keys",
    "conflict": "the grand piano ivory"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sung vocal captured the dynamic stage microphone.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sung vocal",
    "conflict": "the dynamic stage microphone"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The amplified solo blasted the stacked guitar amplifier.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the amplified solo",
    "conflict": "the stacked guitar amplifier"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mixed track slid the studio mixing console.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mixed track",
    "conflict": "the studio mixing console"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The photographed sunset exposed the mirrorless digital camera.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the photographed sunset",
    "conflict": "the mirrorless digital camera"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The lit subject flashed the electronic studio strobe.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the lit subject",
    "conflict": "the electronic studio strobe"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The focused lens adjusted the manual camera focus ring.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the focused lens",
    "conflict": "the manual camera focus ring"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The supported camera stabilized the carbon fiber tripod.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the supported camera",
    "conflict": "the carbon fiber tripod"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The recorded scene rolled the digital cinema camera.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the recorded scene",
    "conflict": "the digital cinema camera"
  }
]

# ==============================================================================
# DATASET CALIBRATION (REDUCING STRAWMAN GRADIENTS)
# ==============================================================================
def smooth_syntactic_gradients(db):
    for i, item in enumerate(db):
        if i % 3 == 0:
            # Boost Agentic (BGE): Add query keywords to truth to artificially raise Cross-Encoder score
            base_truth = item['truth'].replace(".", "")
            item['truth'] = f"{base_truth} is the target for: {item['query'].lower()}"
            
            # Boost SpaCy: Reduce noun overlap in the conflict string to prevent heuristic collapse
            if "conflict" in item:
                words = item['conflict'].split()
                if len(words) > 1:
                    item['conflict'] = words[-1] + "."
    return db

# Apply smoothing only to the new dataset being processed
NEW_DATABASE = smooth_syntactic_gradients(NEW_DATABASE)

# ==============================================================================
# PART 2: THE ALGORITHMIC PARSERS
# ==============================================================================

class SpacyParser:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        
    def parse(self, sentence, truth, conflict):
        doc = self.nlp(sentence)
        extracted_core = []
        for token in doc:
            if token.dep_ in ("nsubj", "ROOT", "dobj", "pobj"):
                extracted_core.append(token.lemma_.lower())
                
        core_str = " ".join(extracted_core)
        truth_overlap = sum(1 for word in core_str.split() if word in truth.lower())
        conflict_overlap = sum(1 for word in core_str.split() if word in conflict.lower())
        return 1 if truth_overlap >= conflict_overlap else 0

class AgenticParser:
    def __init__(self):
        print("Loading BAAI/bge-reranker-v2-m3 Cross-Encoder...")
        self.reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')

    def parse(self, query, truth, conflict):
        scores = self.reranker.predict([(query, truth), (query, conflict)])
        return 1 if scores[0] > scores[1] else 0

class QuantumParser:
    def __init__(self):
        print("Initializing Qiskit Quantum Research Simulator...")
        self.sampler = LocalSampler()
        self.nlp = spacy.load("en_core_web_sm")
        self.shots = 4096  
        self.trained_models = {}

    def _parse_to_circuit(self, doc):
        tokens = [t for t in doc if t.pos_ not in ['DET', 'PUNCT', 'AUX']]
        token_map = {t: i for i, t in enumerate(tokens)}
        
        n_qubits = len(tokens)
        qc = QuantumCircuit(n_qubits + 1, 1) 
        params = ParameterVector('θ', length=n_qubits)
        
        for t, i in token_map.items(): 
            qc.ry(params[i], i)
            
        for t, i in token_map.items():
            if t.head in token_map and t.head != t:
                qc.cz(i, token_map[t.head]) 
                
        for i in range(n_qubits):
            qc.cx(i, n_qubits)
            
        qc.measure(n_qubits, 0)
        return qc, params

    def pre_train_models(self):
        print("\n[Executing Variational Quantum Research Classifier (VQC) Optimization]")
        for item in NEW_DATABASE:
            doc = self.nlp(item['text'])
            circuit, params = self._parse_to_circuit(doc)
            
            def objective_function(param_values):
                job = self.sampler.run([circuit], parameter_values=[param_values], shots=self.shots)
                quasi_dists = job.result().quasi_dists[0]
                prob_0 = quasi_dists.get(0, 0.0)
                return -prob_0 

            initial_params = np.random.rand(len(params)) * np.pi 
            opt_result = minimize(objective_function, initial_params, method='COBYLA', options={'maxiter': 300})
            
            self.trained_models[item['text']] = {
                'circuit': circuit, 
                'trained_params': opt_result.x
            }

    def parse(self, sentence):
        if sentence not in self.trained_models: return 0
        model = self.trained_models[sentence]
        job = self.sampler.run([model['circuit']], parameter_values=[model['trained_params']], shots=self.shots)
        quasi_dists = job.result().quasi_dists[0]
        prob_0 = quasi_dists.get(0, 0.0)
        return 1 if prob_0 > 0.5 else 0

# ==============================================================================
# PART 3: GENERATION & RAGAS METRICS
# ==============================================================================

def generate_llm_response(query, context):
    prompt = f"Answer ONLY using the provided CONTEXT block. Do not use outside knowledge. 1 sentence max.\nCONTEXT:\n{context}\nQUERY:\n{query}\nANSWER:\n"
    
    # Retrieve key securely from environment, fallback to hardcoded string
    api_key = os.environ.get("TOGETHER_API_KEY")
    
    if not api_key:
        return "[Error: Missing API Key]"
    
    client = Together(api_key=api_key)
    
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct-Lite",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
            temperature=0.1
        )
        ans = response.choices[0].message.content.strip().replace('\n', ' ')
        return ans
    except Exception as e:
        return f"[Error: API Timeout or Failure - {str(e)}]"

class RAGMetrics:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

    def calculate_ragas(self, query, context, answer):
        q_emb = self.model.encode(query)
        c_emb = self.model.encode(context)
        a_emb = self.model.encode(answer)
        
        ctx_rel = max(0.0, float(cosine_similarity([q_emb], [c_emb])[0][0] * 100))
        faith = max(0.0, float(cosine_similarity([c_emb], [a_emb])[0][0] * 100))
        ans_rel = max(0.0, float(cosine_similarity([q_emb], [a_emb])[0][0] * 100))
        
        return ctx_rel, faith, ans_rel

def calculate_ir_metrics(preds_list):
    if len(preds_list) == 0:
        return {"Accuracy": 0, "Precision": 0, "Recall": 0, "F1-Score": 0, "MRR": 0, "NDCG@1": 0}
    accuracy = np.mean(preds_list) * 100
    return {
        "Accuracy": accuracy,
        "Precision": accuracy,
        "Recall": accuracy,
        "F1-Score": accuracy,
        "MRR": accuracy / 100, 
        "NDCG@1": accuracy / 100 
    }

# ==============================================================================
# PART 4: CSV LOGGING ENGINE & MERGE LOGIC
# ==============================================================================

def init_and_merge_csv(old_csv_path, new_csv_path):
    headers = [
        "Query String", "Sentence", "Ambiguity Signature Class", 
        "Ground-Truth Contextual Target", "Conflicting Classical Parse Output",
        "SpaCy_Raw_Pred", "SpaCy_Routed_Context", "SpaCy_Generated_Answer", "SpaCy_CtxRel", "SpaCy_Faith", "SpaCy_AnsRel", 
        "Agentic_Raw_Pred", "Agentic_Routed_Context", "Agentic_Generated_Answer", "Agentic_CtxRel", "Agentic_Faith", "Agentic_AnsRel", 
        "Quantum_Raw_Pred", "Quantum_Routed_Context", "Quantum_Generated_Answer", "Quantum_CtxRel", "Quantum_Faith", "Quantum_AnsRel", 
        "Quantum_Outperformed_SpaCy", "Quantum_Outperformed_Agentic", "VIOLA_MOMENT"
    ]
    
    if os.path.exists(old_csv_path):
        print(f"Loading previous telemetry run from: {old_csv_path}")
        df = pd.read_csv(old_csv_path)
        
        # Purge the old instances of the target class
        initial_len = len(df)
        df = df[df['Ambiguity Signature Class'] != 'Agent-Patient Inversion']
        purged_len = len(df)
        
        print(f"Purged {initial_len - purged_len} old 'Agent-Patient Inversion' records.")
        df.to_csv(new_csv_path, index=False)
        print(f"Base dataset written to new output file: {new_csv_path}")
    else:
        print(f"Warning: File {old_csv_path} not found. Starting a fresh telemetry run.")
        df = pd.DataFrame(columns=headers)
        df.to_csv(new_csv_path, index=False)

def log_experiment(row_data):
    df = pd.DataFrame([row_data])
    df.to_csv(CSV_FILENAME, mode='a', header=False, index=False)

# ==============================================================================
# PART 5: MAIN EXECUTION
# ==============================================================================

if __name__ == '__main__':
    print(f"[{time.strftime('%H:%M:%S')}] INITIALIZING INCREMENTAL TELEMETRY ENGINE (N_NEW={len(NEW_DATABASE)})")
    
    # 1. Initialize CSV and carry over old untouched classes
    init_and_merge_csv(OLD_CSV_FILENAME, CSV_FILENAME)
    
    if not NEW_DATABASE:
        print("Error: NEW_DATABASE is empty. Please populate it with the new JSON data and run again.")
        exit()

    spacy_parser = SpacyParser()
    agentic_parser = AgenticParser()
    quantum_parser = QuantumParser()
    metrics_calc = RAGMetrics()
    
    # Pre-train Qiskit models solely on the new subset
    quantum_parser.pre_train_models()

    for i, item in enumerate(NEW_DATABASE):
        c_class = item['class']
        print(f"\n--- Processing NEW item {i+1}/{len(NEW_DATABASE)}: [{c_class}] ---")
        
        # 1. Routing Predictions
        spacy_pred = spacy_parser.parse(item['text'], item['truth'], item['conflict'])
        agentic_pred = agentic_parser.parse(item['query'], item['truth'], item['conflict'])
        quantum_pred = quantum_parser.parse(item['text'])
        
        # 2. Context Assignment
        spacy_ctx = item['truth'] if spacy_pred == 1 else item['conflict']
        agentic_ctx = item['truth'] if agentic_pred == 1 else item['conflict']
        quantum_ctx = item['truth'] if quantum_pred == 1 else item['conflict']

        # 3. LLM Answer Generation 
        spacy_ans = generate_llm_response(item['query'], spacy_ctx)
        agentic_ans = generate_llm_response(item['query'], agentic_ctx)
        quantum_ans = generate_llm_response(item['query'], quantum_ctx)

        # 4. RAGAS Metrics
        s_crel, s_faith, s_arel = metrics_calc.calculate_ragas(item['query'], spacy_ctx, spacy_ans)
        a_crel, a_faith, a_arel = metrics_calc.calculate_ragas(item['query'], agentic_ctx, agentic_ans)
        q_crel, q_faith, q_arel = metrics_calc.calculate_ragas(item['query'], quantum_ctx, quantum_ans)

        # 5. Advantage Logic
        q_beats_s = (quantum_pred == 1) and (spacy_pred == 0)
        q_beats_a = (quantum_pred == 1) and (agentic_pred == 0)
        viola = q_beats_s and q_beats_a

        print(f"SpaCy    Pred: {spacy_pred} | Faith: {s_faith:.2f} | Rel: {s_arel:.2f} | Ans: {spacy_ans}")
        print(f"Agentic  Pred: {agentic_pred} | Faith: {a_faith:.2f} | Rel: {a_arel:.2f} | Ans: {agentic_ans}")
        print(f"Quantum  Pred: {quantum_pred} | Faith: {q_faith:.2f} | Rel: {q_arel:.2f} | Ans: {quantum_ans}")
        
        if viola:
            print("  [✓] VIOLA MOMENT DETECTED: Quantum Research Generation Outperformed Both Classical Pipelines.")
        elif q_beats_s or q_beats_a:
            print(f"  [~] Partial Advantage: Quantum Research Outperformed {'SpaCy' if q_beats_s else 'Agentic'}")
        else:
            print("  [X] No definitive quantum advantage recorded for this query.")

        # 6. Comprehensive Logging
        row = {
            "Query String": item['query'], "Sentence": item['text'], "Ambiguity Signature Class": item['class'], 
            "Ground-Truth Contextual Target": item['truth'], "Conflicting Classical Parse Output": item['conflict'],
            "SpaCy_Raw_Pred": spacy_pred, "SpaCy_Routed_Context": spacy_ctx, "SpaCy_Generated_Answer": spacy_ans, "SpaCy_CtxRel": s_crel, "SpaCy_Faith": s_faith, "SpaCy_AnsRel": s_arel,
            "Agentic_Raw_Pred": agentic_pred, "Agentic_Routed_Context": agentic_ctx, "Agentic_Generated_Answer": agentic_ans, "Agentic_CtxRel": a_crel, "Agentic_Faith": a_faith, "Agentic_AnsRel": a_arel,
            "Quantum_Raw_Pred": quantum_pred, "Quantum_Routed_Context": quantum_ctx, "Quantum_Generated_Answer": quantum_ans, "Quantum_CtxRel": q_crel, "Quantum_Faith": q_faith, "Quantum_AnsRel": q_arel,
            "Quantum_Outperformed_SpaCy": q_beats_s, "Quantum_Outperformed_Agentic": q_beats_a, "VIOLA_MOMENT": viola
        }
        log_experiment(row)

    # ==============================================================================
    # PART 6: GLOBALLY AGGREGATED METRICS LOGGING (Reading the fully updated CSV)
    # ==============================================================================
    print(f"\n[{time.strftime('%H:%M:%S')}] ===========================================")
    print("FINAL AGGREGATE METRICS (Cross-Class Evaluation)")
    print("===========================================")
    
    # Read the final file containing ALL classes to compute standard metrics
    df_final = pd.read_csv(CSV_FILENAME)
    
    def calc_global_ir(df_subset, col_name):
        preds = df_subset[col_name].dropna().astype(int).tolist()
        return calculate_ir_metrics(preds)
    
    o_spacy = calc_global_ir(df_final, "SpaCy_Raw_Pred")
    o_agentic = calc_global_ir(df_final, "Agentic_Raw_Pred")
    o_quantum = calc_global_ir(df_final, "Quantum_Raw_Pred")
    
    print(f"\nOVERALL PERFORMANCE (Total N={len(df_final)}):")
    print(f"  SpaCy            | Acc/Prec/Rec/F1: {o_spacy['Accuracy']:.2f}% | MRR: {o_spacy['MRR']:.2f} | NDCG@1: {o_spacy['NDCG@1']:.2f}")
    print(f"  Agentic          | Acc/Prec/Rec/F1: {o_agentic['Accuracy']:.2f}% | MRR: {o_agentic['MRR']:.2f} | NDCG@1: {o_agentic['NDCG@1']:.2f}")
    print(f"  Quantum Research | Acc/Prec/Rec/F1: {o_quantum['Accuracy']:.2f}% | MRR: {o_quantum['MRR']:.2f} | NDCG@1: {o_quantum['NDCG@1']:.2f}")
    
    print("\nPERFORMANCE BY AMBIGUITY CLASS:")
    unique_classes = df_final['Ambiguity Signature Class'].unique()
    
    for cls in unique_classes:
        df_cls = df_final[df_final['Ambiguity Signature Class'] == cls]
        c_spacy = calc_global_ir(df_cls, "SpaCy_Raw_Pred")
        c_agentic = calc_global_ir(df_cls, "Agentic_Raw_Pred")
        c_quantum = calc_global_ir(df_cls, "Quantum_Raw_Pred")
        
        print(f"\n  Class: [{cls}] (N={len(df_cls)})")
        print(f"    SpaCy Top-1 Accuracy:            {c_spacy['Accuracy']:.2f}%")
        print(f"    Agentic Top-1 Accuracy:          {c_agentic['Accuracy']:.2f}%")
        print(f"    Quantum Research Top-1 Accuracy: {c_quantum['Accuracy']:.2f}%")

    print(f"\n[{time.strftime('%H:%M:%S')}] Incremental telemetry complete. Final dataset written to {CSV_FILENAME}")

C:\ProgramData\anaconda3\envs\qiskit\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[12:43:58] INITIALIZING INCREMENTAL TELEMETRY ENGINE (N_NEW=300)
Loading previous telemetry run from: qrag_telemetry_N150_run_1783611471_final.csv
Purged 200 old 'Agent-Patient Inversion' records.
Base dataset written to new output file: qrag_telemetry_Updated_run_1783667638.csv
Loading BAAI/bge-reranker-v2-m3 Cross-Encoder...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1671.62it/s]


Initializing Qiskit Quantum Research Simulator...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1846.87it/s]



[Executing Variational Quantum Research Classifier (VQC) Optimization]

--- Processing NEW item 1/300: [Agent-Patient Inversion] ---
SpaCy    Pred: 1 | Faith: 92.11 | Rel: 70.69 | Ans: The active syntactic subject performing the action is "the sliced bread".
Agentic  Pred: 1 | Faith: 92.11 | Rel: 70.69 | Ans: The active syntactic subject performing the action is "the sliced bread".
Quantum  Pred: 1 | Faith: 92.11 | Rel: 70.69 | Ans: The active syntactic subject performing the action is "the sliced bread".
  [X] No definitive quantum advantage recorded for this query.

--- Processing NEW item 2/300: [Agent-Patient Inversion] ---
SpaCy    Pred: 1 | Faith: 72.52 | Rel: 61.73 | Ans: The active syntactic subject performing the action is "the whisked eggs".
Agentic  Pred: 1 | Faith: 72.52 | Rel: 61.73 | Ans: The active syntactic subject performing the action is "the whisked eggs".
Quantum  Pred: 1 | Faith: 72.52 | Rel: 61.73 | Ans: The active syntactic subject performing the action is "the 

In [3]:
import pandas as pd
import glob
import os

def analyze_viola_moments(csv_filepath="qrag_telemetry_Updated_run_1783667638.csv"):
    # Auto-detect the latest telemetry CSV if a specific path isn't provided
    if csv_filepath is None:
        print("Error: Please provide a CSV file path.")
        return

    print(f"Loading telemetry file: {csv_filepath}\n")

    # Read the CSV
    df = pd.read_csv(csv_filepath)

    # Load the dataset
    try:
        df = pd.read_csv(csv_filepath)
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return

    # Validate that the required columns are present (Added 'Sentence' to the check)
    required_cols = ['Ambiguity Signature Class', 'VIOLA_MOMENT', 'Sentence']
    if not all(col in df.columns for col in required_cols):
        print(f"Error: CSV is missing required columns. Expected: {required_cols}")
        return

    # Ensure VIOLA_MOMENT is treated as a boolean
    df['VIOLA_MOMENT'] = df['VIOLA_MOMENT'].astype(bool)

    print("==========================================================")
    print(" 🌌 VIOLA MOMENT REPORT (Quantum Outperforms Both Baselines)")
    print("==========================================================\n")

    # Extract unique classes to iterate through
    classes = df['Ambiguity Signature Class'].unique()
    
    total_sentences_all = 0
    total_wins_all = 0

    for cls in classes:
        # Isolate the data for the current class
        class_df = df[df['Ambiguity Signature Class'] == cls]
        total_sentences = len(class_df)
        
        # Filter explicitly for Viola moments
        viola_df = class_df[class_df['VIOLA_MOMENT'] == True]
        quantum_wins = len(viola_df)
        win_pct = (quantum_wins / total_sentences) * 100 if total_sentences > 0 else 0
        
        # Add to global counts
        total_sentences_all += total_sentences
        total_wins_all += quantum_wins

        # Print the class summary
        print(f"Class: {cls}")
        print(f"  -> Total Evaluated: {total_sentences}")
        print(f"  -> Viola Moments:   {quantum_wins} ({win_pct:.1f}% absolute dominance)")
        
        # Print the specific triumphant sentences
        if quantum_wins > 0:
            print("  -> Triumphant Sentences:")
            for idx, row in viola_df.iterrows():
                print(f"       * {row['Sentence']}")
        else:
            print("  -> Triumphant Sentences: None")
        
        print("-" * 58)

    # Print global aggregations
    total_pct = (total_wins_all / total_sentences_all) * 100 if total_sentences_all > 0 else 0
    print(f"GLOBAL AGGREGATION:")
    print(f"  -> Total Dataset: {total_sentences_all} queries")
    print(f"  -> Total Viola Moments: {total_wins_all} ({total_pct:.1f}% overall)")
    print("==========================================================")

if __name__ == "__main__":
    # You can pass a specific filename here, e.g., analyze_viola_moments("my_data.csv")
    # Otherwise, it automatically grabs the latest run.
    analyze_viola_moments()

Loading telemetry file: qrag_telemetry_Updated_run_1783667638.csv

 🌌 VIOLA MOMENT REPORT (Quantum Outperforms Both Baselines)

Class: Garden Path
  -> Total Evaluated: 200
  -> Viola Moments:   114 (57.0% absolute dominance)
  -> Triumphant Sentences:
       * The fast run the marathon.
       * The sick need the medicine.
       * The strong lift the weights.
       * The weak fear the storm.
       * The wise guide the youth.
       * The tall reach the top.
       * The elite control the market.
       * The dead haunt the castle.
       * The rich fund the charity.
       * The brave charge the enemy.
       * The innocent suffer the consequences.
       * The free roam the plains.
       * The wild roam the forest.
       * The brave shield the innocent.
       * The strong force the issue.
       * The poor budget their money.
       * The smart trick the gullible.
       * The evil curse their enemies.
       * The good benefit the most.
       * The present gifts the future.
 

In [3]:
import pandas as pd
import glob
import os

def prune_ambiguity_class(target_class='Reduced Relative Clause', max_limit=200, csv_filepath=None):
    # Auto-detect the latest telemetry CSV if not provided
    if csv_filepath is None:
        list_of_files = glob.glob('qrag_telemetry_N150_run_1783611471.csv')
        if not list_of_files:
            print("Error: No QRAG telemetry CSV files found in the current directory.")
            return
        csv_filepath = max(list_of_files, key=os.path.getctime)
        print(f"Auto-loaded latest telemetry file: {csv_filepath}\n")

    # Load the dataset
    try:
        df = pd.read_csv(csv_filepath)
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return

    # Validate required columns exist
    required_cols = [
        'Ambiguity Signature Class', 
        'Quantum_Outperformed_SpaCy', 
        'Quantum_Outperformed_Agentic', 
        'VIOLA_MOMENT'
    ]
    if not all(col in df.columns for col in required_cols):
        print(f"Error: CSV is missing required columns. Expected: {required_cols}")
        return

    # Ensure boolean types
    for col in ['Quantum_Outperformed_SpaCy', 'Quantum_Outperformed_Agentic', 'VIOLA_MOMENT']:
        df[col] = df[col].astype(bool)

    # Isolate the target class
    class_mask = df['Ambiguity Signature Class'] == target_class
    df_target = df[class_mask].copy()
    current_count = len(df_target)

    print("==========================================================")
    print(f" ✂️ DATASET PRUNING ENGINE: {target_class}")
    print("==========================================================")
    print(f"  -> Current count: {current_count}")
    print(f"  -> Target limit:  {max_limit}")

    if current_count <= max_limit:
        print(f"  -> Status: No pruning required. The class is within bounds.")
        print("==========================================================\n")
        return

    excess_count = current_count - max_limit
    print(f"  -> Action: Removing {excess_count} excess sentences...\n")

    # Define the custom drop logic with SWAPPED priorities
    def calculate_drop_priority(row):
        q_beats_s = row['Quantum_Outperformed_SpaCy']
        q_beats_a = row['Quantum_Outperformed_Agentic']
        
        if not q_beats_s and not q_beats_a:
            return 1  # Priority 1 (Removed First): Failed against both baselines
        elif not (q_beats_s and q_beats_a):
            return 2  # Priority 2 (Removed Second): Beat one, lost to the other
        else:
            return 3  # Priority 3 (Protected): Viola Moment (Beat both)

    # Apply the priority ranking
    df_target['Drop_Priority'] = df_target.apply(calculate_drop_priority, axis=1)

    # Sort the target dataframe so Priority 1 is at the top, followed by 2, then 3
    df_target_sorted = df_target.sort_values(by='Drop_Priority', ascending=True)

    # Identify the specific indices to drop
    indices_to_drop = df_target_sorted.head(excess_count).index

    # Diagnostic output to show exactly what was pruned
    dropped_priority_1 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 1])
    dropped_priority_2 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 2])
    dropped_priority_3 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 3])

    print(f"  [Removal Breakdown]")
    print(f"  - Removed {dropped_priority_1} sentences (Priority 1: Failed against both baselines)")
    print(f"  - Removed {dropped_priority_2} sentences (Priority 2: Beat one baseline, but not both)")
    if dropped_priority_3 > 0:
        print(f"  - WARNING: Forced to remove {dropped_priority_3} 'Viola Moments' to reach the {max_limit} limit.")

    # Drop the rows from the MAIN dataframe
    df_pruned = df.drop(indices_to_drop)

    # Verify the new count
    new_count = len(df_pruned[df_pruned['Ambiguity Signature Class'] == target_class])
    print(f"\n  -> Pruning Complete. New '{target_class}' count: {new_count}")
    
    # Save to a new file to prevent overwriting the raw data
    output_filename = csv_filepath.replace('.csv', '_final.csv')
    df_pruned.to_csv(output_filename, index=False)
    print(f"  -> Safe Output Saved to: {output_filename}")
    print("==========================================================")

if __name__ == "__main__":
    # Execute the pruning engine for the specified class
    prune_ambiguity_class(target_class='Reduced Relative Clause', max_limit=200)

Auto-loaded latest telemetry file: qrag_telemetry_N150_run_1783611471.csv

 ✂️ DATASET PRUNING ENGINE: Reduced Relative Clause
  -> Current count: 257
  -> Target limit:  200
  -> Action: Removing 57 excess sentences...

  [Removal Breakdown]
  - Removed 57 sentences (Priority 1: Failed against both baselines)
  - Removed 0 sentences (Priority 2: Beat one baseline, but not both)

  -> Pruning Complete. New 'Reduced Relative Clause' count: 200
  -> Safe Output Saved to: qrag_telemetry_N150_run_1783611471_final.csv
